# Banking analytics · Source → Bronze → Silver
Use the existing repository locally or in Colab. This notebook calls production modules and never builds Gold or Power BI. Previews omit customer IDs.

Set `PROJECT` below if Colab opened this notebook outside the extracted/cloned repository. Upload the repository first through the Colab Files pane or clone your own repository. No public URL is invented.

In [ ]:
from pathlib import Path
import os, sys, json
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('OMP_NUM_THREADS', '1')
# In Colab: clone/extract this existing repository, then set PROJECT to its directory.
PROJECT = Path.cwd()
if PROJECT.name == 'notebooks': PROJECT = PROJECT.parent
if not (PROJECT / 'requirements-pipeline.txt').exists():
    raise FileNotFoundError('Set PROJECT to the existing repository root before continuing.')
os.chdir(PROJECT)
if str(PROJECT) not in sys.path: sys.path.insert(0, str(PROJECT))


In [ ]:
%pip install -q -r requirements-pipeline.txt

In [ ]:
import pandas as pd
from src.utils.config import load_config, location
from src.ingestion.source_validator import validate_source
from src.ingestion.bronze_ingestion import read_bronze
from src.pipeline import run
config = load_config()


Optional Excel upload. The supplied workbook uses sheet `Total Data`, zero-based header row 1. For a different layout edit `config/config.yaml` before ingestion. Column normalization handles syntax only; semantic aliases must be explicit.

In [ ]:
UPLOAD_EXCEL = False
SOURCE_OVERRIDE = None
if UPLOAD_EXCEL:
    from google.colab import files
    uploaded = files.upload()
    candidates = [name for name in uploaded if name.lower().endswith(('.xlsx', '.xlsm'))]
    if len(candidates) != 1: raise ValueError('Upload exactly one Excel file.')
    import shutil, uuid
    destination = location(config, 'raw') / (uuid.uuid4().hex + '.xlsx')
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(candidates[0], destination)
    SOURCE_OVERRIDE = destination
source = SOURCE_OVERRIDE or location(config, 'source')
validation = validate_source(source, config)
display(validation.as_dict())

Bronze preserves business cell values and adds UTC metadata. Mixed cell types use reversible tagged storage encoding; `read_bronze` restores the original values. No cleaning or deduplication happens here.

In [ ]:
batch = run(stage='bronze', source=SOURCE_OVERRIDE)
bronze = read_bronze(config, batch)
print('Bronze rows:', len(bronze))
display(bronze.drop(columns='customer_id').head())

Silver standardizes, validates, creates bands and routes critical records to quarantine. Missing numeric inputs are warnings and remain null. The earliest physical source row wins duplicate IDs even when a later duplicate has more complete values.

In [ ]:
run(stage='silver', batch_id=batch)
manifest = json.loads((location(config, 'reports') / 'latest_silver.json').read_text())
silver = pd.read_parquet(Path(manifest['silver_path']) / 'silver_customers.parquet')
quarantine = pd.read_parquet(Path(manifest['quarantine_path']) / 'silver_customers_quarantine.parquet')
quality = pd.read_parquet(Path(manifest['quality_path']) / 'data_quality_report.parquet')
display(quality)
print({'Bronze': len(bronze), 'Accepted Silver': len(silver), 'Quarantine': len(quarantine)})
assert len(bronze) == len(silver) + len(quarantine)
display(silver.drop(columns=['customer_id', 'original_customer_id']).head())
display(quarantine[['rejection_reason', 'quality_issues', 'data_quality_issue_count', '_source_row_number']].head())

In [ ]:
summary = json.loads((Path(manifest['quality_path']) / 'exploration_summary.json').read_text())
display(summary)

Parquet exports are already persisted in the manifest paths. Optional CSV files below contain customer data; export them only to an approved environment.

In [ ]:
EXPORT_CSV = False
if EXPORT_CSV:
    export = PROJECT / 'outputs' / batch
    export.mkdir(parents=True, exist_ok=True)
    silver.to_csv(export / 'silver_customers.csv', index=False)
    quarantine.to_csv(export / 'silver_quarantine.csv', index=False)
    quality.to_csv(export / 'data_quality_report.csv', index=False)
    print(export)